# SAP effective-invoice reconciliation
Read-only audit, 6 October 2026 (Australia/Sydney). No dashboard files changed.
Sources: SAP VBRK headers and VBRP service invoice items; saved SAPInvoiceLookup; original September XLSX filtered to SOWTY999/SOWTY777. Queries and snapshots are saved beside this notebook.
Compare monthly signed billing-document movements to invoices still valid at extraction time. Exclude S1 cancellation documents and originals marked FKSTO=X or referenced by SFAKN. Use the surviving invoice billing date; do not shift it to an older invoice month.
All audited currency values are AUD. No multiple currently valid service invoices per sales order were found in this scope. Cross-month cancellation changes monthly totals even when the year total reconciles. Counts exclude C4C missing-invoice placeholders.

In [1]:
import json
from decimal import Decimal
from pathlib import Path
base = Path.cwd() if Path("sap_headers.json").exists() else Path("outputs/invoice-validity-audit")
headers = json.loads((base / "sap_headers.json").read_text())
results = json.loads((base / "results.json").read_text())
originals_cancelled = {row["SFAKN"] for row in headers if row["SFAKN"]}
active = [row for row in headers if row["FKART"] == "F2" and row["FKSTO"] != "X" and row["VBELN"] not in originals_cancelled]
assert all(Decimal(row["NETWR"]) >= 0 for row in active)
for prefix in ("202609", "2026"):
    rows = [row for row in active if row["FKDAT"].startswith(prefix)]
    total = sum((Decimal(row["NETWR"]) for row in rows), Decimal(0))
    print(prefix, "valid documents:", len(rows), "AUD:", total)
print("Period | Signed net | Currently valid | Difference")
for period, row in results["saved_months"].items():
    net = Decimal(row["signed_total"])
    valid = Decimal(row["valid_total"])
    print(period, net, valid, valid-net, sep=" | ")
assert results["file_september"]["valid_total"] == "48442.32"
assert results["saved_september"]["valid_total"] == "48492.32"
assert results["saved_2026"]["valid_total"] == "478916.87"


202609 valid documents: 124 AUD: 48492.32
2026 valid documents: 1095 AUD: 478916.87
Period | Signed net | Currently valid | Difference
202601 | 16272.50 | 16272.50 | 0.00
202602 | 16357.18 | 16357.18 | 0.00
202603 | 62740.24 | 62740.24 | 0.00
202604 | 30470.85 | 29888.35 | -582.50
202605 | 98931.42 | 98638.28 | -293.14
202606 | 84149.89 | 82816.28 | -1333.61
202607 | 63115.29 | 61489.95 | -1625.34
202608 | 55268.06 | 59102.65 | 3834.59
202609 | 48492.32 | 48492.32 | 0.00
202610 | 3119.12 | 3119.12 | 0.00
